# Phase 3 - Notebook 06: DUSt3R Inference Pipeline Code Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/06_code_walkthrough.ipynb)


## Setup


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from src.dust3r.pointmap import PointMap
from src.dust3r.alignment import GlobalAligner
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print("Setup complete!")

## 1. DUSt3R 推理管道概览

```
加载图像
    ↓
加载预训练模型
    ↓
对每个图像对:
  ├─ 前向传播
  ├─ 提取 Pointmaps + Confidence
  ├─ 过滤低置信度点
  └─ 估计相对位姿
    ↓
全局对齐
    ↓
合并为全局点云
    ↓
保存/可视化
```


## 2. 步骤 1：加载和预处理图像


In [ ]:
def load_and_preprocess_image(image_path, target_size=512):
    """
    加载图像并预处理为网络输入格式。
    
    Args:
        image_path: 图像文件路径
        target_size: 目标分辨率
    
    Returns:
        tensor: [1, 3, H, W] 张量
    """
    # 在实际中使用 PIL 或 OpenCV
    # 这里演示合成图像
    
    img = np.random.rand(target_size, target_size, 3) * 0.5 + 0.25
    # 标准化 [0, 1]
    img = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0)
    return img

# 演示
img1 = load_and_preprocess_image('image1.jpg')
img2 = load_and_preprocess_image('image2.jpg')

print(f"图像 1: {img1.shape}")
print(f"图像 2: {img2.shape}")
print(f"值范围: [{img1.min():.3f}, {img1.max():.3f}]")

## 3. 步骤 2：前向传播


In [ ]:
def dust3r_inference(model, img1, img2):
    """
    DUSt3R 前向推理。
    
    Args:
        model: DUSt3R 模型
        img1: [B, 3, H, W]
        img2: [B, 3, H, W]
    
    Returns:
        dict: 包含 Pointmaps, Confidence, 等
    """
    with torch.no_grad():
        # ViT 编码
        feat1 = model.encoder(img1)  # [B, N, D]
        feat2 = model.encoder(img2)
        
        # 交叉注意力解码
        feat1_dec, feat2_dec = model.decoder(feat1, feat2)
        
        # Pointmap 预测
        points1, conf1 = model.head1(feat1_dec)  # [B, H, W, 3], [B, H, W]
        points2, conf2 = model.head2(feat2_dec)
        
        return {
            'points1': points1.cpu().numpy(),
            'points2': points2.cpu().numpy(),
            'conf1': conf1.cpu().numpy(),
            'conf2': conf2.cpu().numpy(),
        }

print("dust3r_inference 函数定义完成")

## 4. 步骤 3：Pointmap 创建和过滤


In [ ]:
# 创建合成的 DUSt3R 输出
H, W = 64, 64
points1 = np.random.randn(H, W, 3) * 0.5 + np.array([0, 0, 2])[None, None, :]
conf1 = 0.7 + 0.3 * np.random.rand(H, W)

points2 = np.random.randn(H, W, 3) * 0.5 + np.array([1, 0, 2])[None, None, :]
conf2 = 0.7 + 0.3 * np.random.rand(H, W)

# 创建 Pointmaps
pm1 = PointMap(points1, confidence=conf1)
pm2 = PointMap(points2, confidence=conf2)

print(f"Pointmap 1: {pm1}")
print(f"Pointmap 2: {pm2}")

# 过滤低置信度
pm1_filtered = pm1.filter_by_confidence(threshold=0.5)
pm2_filtered = pm2.filter_by_confidence(threshold=0.5)

print(f"\n过滤后的点数: {(pm1_filtered.confidence >= 0.5).sum()} / {H * W}")

## 5. 步骤 4：相对位姿估计


In [ ]:
def estimate_relative_pose(pm1, pm2):
    """
    使用 Procrustes 从两个 Pointmaps 估计相对位姿。
    """
    pts1 = pm1.points.reshape(-1, 3)
    pts2 = pm2.points.reshape(-1, 3)
    
    # 仅使用高置信度点
    conf1_flat = pm1.confidence.flatten()
    mask = conf1_flat > 0.5
    
    pts1_good = pts1[mask]
    pts2_good = pts2[mask]
    
    if len(pts1_good) < 10:
        print("警告: 置信度高的点太少")
        return None, None
    
    # Procrustes 对齐
    R, t = GlobalAligner.procrustes_align(pts1_good, pts2_good)
    
    return R, t

R_rel, t_rel = estimate_relative_pose(pm1, pm2)

if R_rel is not None:
    print(f"估计的相对位姿:")
    print(f"  旋转行列式 (应为 1.0): {np.linalg.det(R_rel):.3f}")
    print(f"  平移: {t_rel}")

## 6. 步骤 5：全局对齐和融合


In [ ]:
# 多视图场景
aligner = GlobalAligner()
aligner.add_frame('frame_0', pm1)
aligner.add_frame('frame_1', pm2)

pairwise_poses = {('frame_0', 'frame_1'): (R_rel, t_rel)}
global_poses = aligner.pairwise_to_global(pairwise_poses)

print("全局位姿:")
for frame_id, (R, t) in global_poses.items():
    print(f"  {frame_id}: t = {t}")

## 7. 完整推理管道


In [ ]:
# 完整推理管道演示
print("=" * 60)
print("DUSt3R 完整推理管道")
print("=" * 60)
print()
print("Step 1: 加载图像对")
print(f"  Image 1: {img1.shape}")
print(f"  Image 2: {img2.shape}")
print()
print("Step 2: DUSt3R 前向传播")
print(f"  ViT Encoder → Features")
print(f"  Cross-Attn Decoder → Fused features")
print(f"  Output heads → Pointmaps + Confidence")
print()
print("Step 3: 创建 PointMap 对象")
print(f"  PointMap 1: {pm1}")
print(f"  PointMap 2: {pm2}")
print()
print("Step 4: 置信度过滤")
print(f"  高置信度点 (>0.5): {(pm1.confidence > 0.5).sum()} / {H*W}")
print()
print("Step 5: 相对位姿估计")
if R_rel is not None:
    print(f"  R: 旋转矩阵 [3×3]")
    print(f"  t: 平移向量 {t_rel}")
print()
print("Step 6: 全局对齐")
print(f"  Frame count: {len(aligner.poses)}")
print()
print("Step 7: 点云融合")
pts1, _ = pm1.to_pointcloud(0.5)
pts2, _ = pm2.to_pointcloud(0.5)
print(f"  总点数: {len(pts1) + len(pts2)}")
print()
print("=" * 60)

## 8. Summary

**推理管道关键步骤:**
1. 图像预处理
2. ViT 编码 + 交叉注意力解码
3. Pointmap 和置信度预测
4. 置信度过滤
5. Procrustes 位姿估计
6. 全局对齐
7. 点云融合

---

**Next**: [07_custom_data.ipynb](./07_custom_data.ipynb)
